# Loading train data

In [3]:
import pandas as pd
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
%matplotlib inline
train_data = pd.read_csv("train.csv")

In [4]:
train_data.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [5]:
train_data.shape

(8523, 12)

In [6]:
train_data.dtypes

Item_Identifier               object
Item_Weight                  float64
Item_Fat_Content              object
Item_Visibility              float64
Item_Type                     object
Item_MRP                     float64
Outlet_Identifier             object
Outlet_Establishment_Year      int64
Outlet_Size                   object
Outlet_Location_Type          object
Outlet_Type                   object
Item_Outlet_Sales            float64
dtype: object

In [7]:
train_data.isnull().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [8]:
train_data.columns

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Outlet_Sales'],
      dtype='object')

In [9]:
cat_columns = [ 'Item_Fat_Content', 
       'Item_Type', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type']
for val in  cat_columns:
    print(train_data[val].unique(),"***********",val)

['Low Fat' 'Regular' 'low fat' 'LF' 'reg'] *********** Item_Fat_Content
['Dairy' 'Soft Drinks' 'Meat' 'Fruits and Vegetables' 'Household'
 'Baking Goods' 'Snack Foods' 'Frozen Foods' 'Breakfast'
 'Health and Hygiene' 'Hard Drinks' 'Canned' 'Breads' 'Starchy Foods'
 'Others' 'Seafood'] *********** Item_Type
['OUT049' 'OUT018' 'OUT010' 'OUT013' 'OUT027' 'OUT045' 'OUT017' 'OUT046'
 'OUT035' 'OUT019'] *********** Outlet_Identifier
[1999 2009 1998 1987 1985 2002 2007 1997 2004] *********** Outlet_Establishment_Year
['Medium' nan 'High' 'Small'] *********** Outlet_Size
['Tier 1' 'Tier 3' 'Tier 2'] *********** Outlet_Location_Type
['Supermarket Type1' 'Supermarket Type2' 'Grocery Store'
 'Supermarket Type3'] *********** Outlet_Type


# Handling the null values of columns

In [10]:
train_data["Outlet_Size"].fillna(train_data["Outlet_Size"].mode()[0],inplace=True)

In [11]:
train_data["Item_Weight"].fillna(train_data["Item_Weight"].mean(),inplace=True)

In [12]:
train_data["Outlet_Establishment_Year"] =train_data["Outlet_Establishment_Year"].astype(str) 

# Preprocessing categorical feature

In [13]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

cat_columns = [ 'Item_Fat_Content', 
       'Item_Type', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type']

for val in  cat_columns:
    train_data[val]=label_encoder.fit_transform(train_data[val])


In [14]:
X= train_data.drop(columns=["Item_Identifier","Item_Outlet_Sales"])
y= train_data["Item_Outlet_Sales"]
X_train,X_cv,y_train,y_cv = train_test_split(X,y,random_state=10,test_size=0.2)

In [19]:
X_train.shape

(6818, 10)

# Model Building

In [55]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense,Dropout
from keras.callbacks import EarlyStopping

In [56]:
# Build the Keras model
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model = Sequential()
model.add(Dense(128, input_dim=10, activation='relu'))
#model.add(Dropout(0.5))  # Add dropout for regularization
model.add(Dense(64, activation='relu'))
#model.add(Dropout(0.3))  # Add dropout for regularization
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='linear'))   # Output layer with linear activation for regression

# Compile the model
model.compile(loss='mean_squared_error', optimizer='adam',metrics=['mae', 'mape'])

# Train the model
model.fit(X_train, y_train, epochs=200, batch_size=16, validation_data=(X_cv, y_cv),callbacks=[early_stopping])


Epoch 1/200
427/427 [==============================] - 3s 4ms/step - loss: 2933064.5000 - mae: 1232.8472 - mape: 137.4841 - val_loss: 1892341.5000 - val_mae: 1030.9164 - val_mape: 157.3933
Epoch 2/200
427/427 [==============================] - 1s 3ms/step - loss: 1930394.7500 - mae: 1026.1442 - mape: 145.2298 - val_loss: 1829333.6250 - val_mae: 990.7149 - val_mape: 131.6454
Epoch 3/200
427/427 [==============================] - 1s 3ms/step - loss: 1841045.1250 - mae: 1002.0168 - mape: 135.7269 - val_loss: 1770386.0000 - val_mae: 1003.0430 - val_mape: 146.5498
Epoch 4/200
427/427 [==============================] - 1s 3ms/step - loss: 1725346.2500 - mae: 972.9648 - mape: 123.1899 - val_loss: 1658895.7500 - val_mae: 979.9299 - val_mape: 133.1014
Epoch 5/200
427/427 [==============================] - 1s 3ms/step - loss: 1616959.6250 - mae: 953.4192 - mape: 110.6982 - val_loss: 1517218.5000 - val_mae: 931.6189 - val_mape: 109.3948
Epoch 6/200
427/427 [==============================] - 1s 3m

427/427 [==============================] - 1s 3ms/step - loss: 1176362.6250 - mae: 763.2025 - mape: 55.8858 - val_loss: 1144282.6250 - val_mae: 748.7341 - val_mape: 55.4345
Epoch 46/200
427/427 [==============================] - 1s 3ms/step - loss: 1175622.7500 - mae: 761.2576 - mape: 55.4616 - val_loss: 1153410.8750 - val_mae: 758.9897 - val_mape: 58.3955
Epoch 47/200
427/427 [==============================] - 1s 3ms/step - loss: 1178848.3750 - mae: 762.9459 - mape: 55.2390 - val_loss: 1146911.8750 - val_mae: 749.8068 - val_mape: 52.6649
Epoch 48/200
427/427 [==============================] - 1s 3ms/step - loss: 1182181.6250 - mae: 764.7332 - mape: 56.3195 - val_loss: 1158222.1250 - val_mae: 763.5031 - val_mape: 60.7427
Epoch 49/200
427/427 [==============================] - 1s 3ms/step - loss: 1178178.6250 - mae: 763.0427 - mape: 55.3981 - val_loss: 1148519.5000 - val_mae: 752.6644 - val_mape: 56.6204
Epoch 50/200
427/427 [==============================] - 1s 3ms/step - loss: 1168825

# Now evalutaing test data 

In [57]:
test_data = pd.read_csv("test.csv")
test_data.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,FDW58,20.750,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1
1,FDW14,8.300,reg,0.038428,Dairy,87.3198,OUT017,2007,NaN,Tier 2,Supermarket Type1
2,NCN55,14.600,Low Fat,0.099575,Others,241.7538,OUT010,1998,NaN,Tier 3,Grocery Store
3,FDQ58,7.315,Low Fat,0.015388,Snack Foods,155.0340,OUT017,2007,NaN,Tier 2,Supermarket Type1
4,FDY38,NaN,Regular,0.118599,Dairy,234.2300,OUT027,1985,Medium,Tier 3,Supermarket Type3


In [58]:
test_data.isnull().sum()

Item_Identifier                 0
Item_Weight                   976
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1606
Outlet_Location_Type            0
Outlet_Type                     0
dtype: int64

# Handling null value of test data

In [59]:
test_data["Outlet_Size"].fillna(test_data["Outlet_Size"].mode()[0], inplace=True)
test_data["Item_Weight"].fillna(test_data["Item_Weight"].mean(), inplace=True)
test_data["Outlet_Establishment_Year"] = test_data["Outlet_Establishment_Year"].astype(str)

# preprocessing of test data

In [60]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

cat_columns = [ 'Item_Fat_Content', 
       'Item_Type', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type']

for val in  cat_columns:
    test_data[val]=label_encoder.fit_transform(test_data[val])


In [61]:
X_test = test_data.drop(columns=["Item_Identifier"])

In [62]:
predictions = model.predict(X_test)

178/178 [==============================] - 0s 2ms/step


In [64]:
test_data["Item_Outlet_Sales_Pred"] = predictions

In [65]:
sam_sub=pd.read_csv("sample_submission.csv")

In [66]:
sam_sub["Item_Outlet_Sales"] = predictions

In [67]:
sam_sub.to_csv("sample_sub_pred.csv",index=False)

In [68]:
test = pd.read_csv("sample_sub_pred.csv")

In [69]:
test.head()

,Item_Identifier,Outlet_Identifier,Item_Outlet_Sales
0,FDW58,OUT049,1709.20780
1,FDW14,OUT017,1484.22450
2,NCN55,OUT010,562.63477
3,FDQ58,OUT017,2533.15620
4,FDY38,OUT027,5987.64940


In [63]:
count=0
for val in predictions :
    if val < 0:
        print(val)
        count=count+1
print(count)        

0


In [ ]:
coun